# Phase 3 — Model Training

Trains TF-IDF + LR and TF-IDF + SVM on the cleaned, split dataset and saves the best model to `models/classifier.pkl`.

Inputs: `data/processed/train.csv`, `data/processed/val.csv`
Output: `models/classifier.pkl`

In [1]:
!pip install scikit-learn pandas joblib --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import f1_score, classification_report

# Run from project root (works whether notebook is opened in notebooks/ or root)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

TRAIN = "data/processed/train.csv"
VAL = "data/processed/val.csv"
MODEL_OUT = "models/classifier.pkl"

cwd: C:\Users\Feras\Desktop\complaint classifier


## Load data

In [3]:
train = pd.read_csv(TRAIN)
val = pd.read_csv(VAL)
print('train:', len(train), 'val:', len(val))
print()
print('per-category counts (train):')
print(train['category'].value_counts())

train: 68826 val: 13277

per-category counts (train):
category
جودة الطعام      31558
السعر والقيمة    12173
خدمة الموظفين    10669
وقت الانتظار      2963
دقة الطلب         2458
النظافة           2408
الجو والمكان      2329
التوصيل           2197
عامة              2071
Name: count, dtype: int64


## Baseline (sanity check)

Majority class baseline — the model must beat this.

In [4]:
majority_class = train['label'].mode()[0]
baseline_acc = (val['label'] == majority_class).mean()
print(f'Majority-class baseline accuracy on val: {baseline_acc:.3f}')

Majority-class baseline accuracy on val: 0.509


## Build feature pipeline

Word n-grams capture phrases. Char n-grams capture Arabic morphology — sklearn treats `أكل / الأكل / الأكلة` as different word tokens, but they share character roots.

In [5]:
def build_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), analyzer='word')),
        ('char', TfidfVectorizer(max_features=20000, ngram_range=(3, 5), analyzer='char_wb')),
    ])

## Train Logistic Regression

In [6]:
lr = Pipeline([
    ('feat', build_features()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)),
])
lr.fit(train['text'], train['label'])
lr_pred = lr.predict(val['text'])
lr_f1 = f1_score(val['label'], lr_pred, average='weighted')
print(f'LR weighted F1: {lr_f1:.4f}')

C:\Users\Feras\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LR weighted F1: 0.8619


## Train Linear SVM

In [7]:
svm = Pipeline([
    ('feat', build_features()),
    ('clf', LinearSVC(class_weight='balanced')),
])
svm.fit(train['text'], train['label'])
svm_pred = svm.predict(val['text'])
svm_f1 = f1_score(val['label'], svm_pred, average='weighted')
print(f'SVM weighted F1: {svm_f1:.4f}')

SVM weighted F1: 0.8980


## Per-class breakdown

In [8]:
label_map = json.load(open('data/processed/label_map.json'))
inv = {v: k for k, v in label_map.items()}
names = [inv[i] for i in range(len(inv))]

print('=== LR per-class ===')
print(classification_report(val['label'], lr_pred, target_names=names, zero_division=0))
print('=== SVM per-class ===')
print(classification_report(val['label'], svm_pred, target_names=names, zero_division=0))

=== LR per-class ===
               precision    recall  f1-score   support

      التوصيل       0.50      0.73      0.59       151
 الجو والمكان       0.19      0.31      0.23        85
السعر والقيمة       0.89      0.87      0.88      2608
      النظافة       0.71      0.91      0.79       516
  جودة الطعام       0.95      0.83      0.89      6762
خدمة الموظفين       0.82      0.94      0.88      2286
    دقة الطلب       0.29      0.25      0.27        20
         عامة       0.46      0.84      0.59       214
 وقت الانتظار       0.71      0.92      0.80       635

     accuracy                           0.86     13277
    macro avg       0.61      0.73      0.66     13277
 weighted avg       0.88      0.86      0.86     13277

=== SVM per-class ===
               precision    recall  f1-score   support

      التوصيل       0.59      0.56      0.57       151
 الجو والمكان       0.28      0.21      0.24        85
السعر والقيمة       0.91      0.90      0.91      2608
      النظافة     

## Pick winner and save

Note: Logistic Regression is preferred when F1 is close (within 1%) because it provides `predict_proba` — needed for the top-3 confidence demo in Phase 5.

In [9]:
os.makedirs('models', exist_ok=True)
if lr_f1 >= svm_f1 - 0.01:
    best, name, best_f1 = lr, 'LogisticRegression', lr_f1
else:
    best, name, best_f1 = svm, 'LinearSVC', svm_f1
joblib.dump(best, MODEL_OUT)
print(f'Saved {name} (val weighted F1 = {best_f1:.4f}) -> {MODEL_OUT}')

Saved LinearSVC (val weighted F1 = 0.8980) -> models/classifier.pkl


## Sanity check predictions

In [10]:
samples = [
    'وصل الطلب بارد جدا والمندوب تاخر اكثر من ساعتين',
    'الاسعار مبالغ فيها لا تناسب الجوده المقدمه',
    'الديكور قديم والكراسي غير مريحه والمطعم مزدحم',
    'طلبت برجر بدون بصل لكنهم وضعوه رغم تنبيهي',
    'الاكل بارد وبدون نكهة',
    'النظافه سيئه الطاولات متسخه والارض غير نظيفه',
]
for s in samples:
    pred = best.predict([s])[0]
    print(f'{inv[pred]:>20s}  <-  {s}')

             التوصيل  <-  وصل الطلب بارد جدا والمندوب تاخر اكثر من ساعتين
       السعر والقيمة  <-  الاسعار مبالغ فيها لا تناسب الجوده المقدمه
        الجو والمكان  <-  الديكور قديم والكراسي غير مريحه والمطعم مزدحم
           دقة الطلب  <-  طلبت برجر بدون بصل لكنهم وضعوه رغم تنبيهي
         جودة الطعام  <-  الاكل بارد وبدون نكهة
             النظافة  <-  النظافه سيئه الطاولات متسخه والارض غير نظيفه
